# Forecasting — full SMOA bằng Python

Chạy các cell theo thứ tự bằng kernel **`.venv`** của project.
Notebook dùng lại thuật toán trong các module Python, để kết quả thống nhất với
pipeline full: **BO winner → UQ calibration → point/quantile prediction**.

Mặc định `FORECAST_MODE="pipeline"`: đọc tiến độ và kết quả của job full hiện tại.
Chọn `"run"` để chạy forecasting riêng sau khi BO hoàn tất; output riêng nằm ở
`outputs/notebook_forecasting/full`, dùng đủ 50 bang và mọi origin đủ điều kiện.
Không chạy lại generation/BO khi mở notebook.

**Lưu ý phiên bản:** artifacts `outputs/reproduction_full` đang hiển thị là
run legacy **final truth + spline**. Code mới mặc định dùng thin plate GAM/GCV.
Xem [Experiment 0](../reproduction/experiment_0.ipynb) để đọc kết quả đã chạy
lại với historical JHU và forecast COVIDHub gốc; không đồng nhất hai run.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import json
import os
import subprocess
import sys
import tempfile

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "experiments/run_pipeline.py").is_file())
for folder in ["experiments", "experiments/forecasting"]:
    sys.path.insert(0, str(ROOT / folder))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "smoa-notebook-matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

pd.set_option("display.max_columns", 15)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

def read_json(path):
    return json.loads(Path(path).read_text()) if Path(path).is_file() else {}

def show_figure(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(data=buffer.getvalue()))

print("Python:", sys.executable)
print("Snapshot UTC:", datetime.now(timezone.utc).isoformat())

## 1. Cấu hình

`SELECTED_SERIES` chỉ chọn bang để **hiển thị**, không cắt số bang của thực nghiệm.
`START_DATE` là forecast origin đầu tiên; h=4, NB 5.000 draws, seed=13 theo pipeline.
Với archive lịch sử, điền đường dẫn vào `as_of_truth` trong manifest.

In [ ]:
RUN_DIR = ROOT / "outputs/reproduction_full"
MANIFEST = ROOT / "outputs/truth_states/manifest.csv"
SELECTED_SERIES = "California"
FORECAST_MODE = "pipeline"  # "pipeline": xem job hiện tại; "run": chạy riêng sau BO
NOTEBOOK_OUTPUT = ROOT / "outputs/notebook_forecasting/full"
START_DATE = "2020-08-15"
SEED = 13

## 2. BO đã hoàn tất chưa?

Chỉ dùng `best_params.json` và selected library khi metadata BO có `status=complete`.
Các candidates đang chạy chưa phải winner; không thay bằng tham số demo/default.

In [ ]:
if FORECAST_MODE not in {"pipeline", "run"}:
    raise ValueError("FORECAST_MODE phải là pipeline hoặc run")
SELECTION_DIR = RUN_DIR / "02_optimization"
bo_config = read_json(SELECTION_DIR / "config.json")
bo_metadata = read_json(SELECTION_DIR / "metadata.json")
bo_checkpoint = read_json(SELECTION_DIR / "checkpoint.json")
bo_progress = read_json(SELECTION_DIR / "progress.json")
BO_READY = bo_metadata.get("status") == "complete"
completed = len(bo_checkpoint.get("history", []))
budget = bo_config.get("initial_points", 99) + bo_config.get("iterations", 5)
display(pd.DataFrame([{
    "BO status": bo_metadata.get("status", "not started"),
    "completed trials": completed, "budget": budget,
    "full search budget": bo_config.get("full_search_budget"),
    "phase": bo_progress.get("phase", "not started"),
}]))
if BO_READY:
    selected = read_json(SELECTION_DIR / "best_params.json")
    display(pd.DataFrame([selected]))
    display(pd.DataFrame([read_json(SELECTION_DIR / "best_library/metadata.json")]))
else:
    selected = None
    print("BO chưa hoàn tất. Chưa có selected library để phát hành dự báo mới.")

## 3. COVID truth và manifest

Đây là dữ liệu COVID thực tế. Cột `as_of_truth` trống nghĩa là history lấy từ
final truth; không gọi dữ liệu này là historical vintages. Truth tương lai
không được truyền vào API `predict_origin`.

In [ ]:
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
if manifest.empty or manifest["series"].duplicated().any():
    raise ValueError("Manifest phải có các bang duy nhất")
selection = manifest.loc[manifest["series"] == SELECTED_SERIES]
if len(selection) != 1:
    raise ValueError(f"Không tìm thấy đúng một bang {SELECTED_SERIES}")
entry = selection.iloc[0]
TRUTH = (MANIFEST.parent / entry["truth"]).resolve()
truth = pd.read_csv(TRUTH, parse_dates=["target_end_date"])
history_mode = "as_of" if manifest["as_of_truth"].ne("").all() else "final"
if manifest["as_of_truth"].ne("").any() and history_mode != "as_of":
    raise ValueError("Cần as_of_truth cho tất cả bang hoặc để trống tất cả")
print(f"Manifest: {len(manifest)} bang; history mode: {history_mode}")
display(manifest.head())
display(truth.tail())
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(truth["target_end_date"], truth["value"], color="#172537", linewidth=1.5)
ax.set(title=f"{SELECTED_SERIES} | Observed COVID-19 final truth (input)",
       xlabel="Target week ending", ylabel="Weekly reported cases")
ax.grid(axis="y", alpha=.2)
show_figure(fig)

## 4. UQ trước prediction tại mỗi origin

6 origins đầu dùng NB size khởi tạo; sau đó fit riêng từng horizon từ forecast
cũ đã có outcome. Bảng dưới là **chính sách khởi tạo**, không phải MLE đã fit.
`source` protocol dùng final labels cho calibration và frozen prehistory
2020-08-15, tương ứng pipeline hiện tại.

In [ ]:
from uncertainty import INITIAL_SIZES, QUANTILES, LONG_QUANTILES

display(pd.DataFrame({"horizon (weeks)": range(1, 5), "initial NB size": INITIAL_SIZES}))
print("Predictive quantiles:", len(LONG_QUANTILES), "| WIS quantiles:", QUANTILES.tolist())
policy = read_json(RUN_DIR / "03_uq/policy.json")
if policy:
    display(pd.DataFrame([{
        "calibration truth": policy["calibration_truth"],
        "initial origins": policy["initial_origins"],
        "execution": policy["execution"],
    }]))
else:
    print("Bước UQ của job full chưa bắt đầu. Các fitted sizes sẽ nằm trong calibration.csv.")

## 5. Chạy forecasting hoặc đọc kết quả pipeline

Trong chế độ `run`, cell này gọi runner với **selected library**, đủ mọi bang,
`max-origins=0`, 5.000 NB draws và 23 quantiles. Chạy lại tiếp tục bằng `--resume`.
`pipeline` chỉ đọc kết quả đã được job hiện tại ghi, không tạo một lượt tính toán khác.

In [ ]:
PREDICTION_DIR = RUN_DIR / "04_prediction" if FORECAST_MODE == "pipeline" else NOTEBOOK_OUTPUT
if FORECAST_MODE == "run" and BO_READY:
    command = [sys.executable, "-u", str(ROOT / "experiments/forecasting/run_benchmark.py"),
        "--manifest", str(MANIFEST), "--selection-dir", str(SELECTION_DIR),
        "--history-mode", history_mode, "--probabilistic", "--max-origins", "0",
        "--start-date", START_DATE, "--warmup-start-date", "1900-01-01",
        "--prehistory-mode", "frozen", "--calibration-truth", "final",
        "--score-truth", "raw", "--coverage-grid", "full", "--negative-truth", "clip",
        "--seed", str(SEED), "--interval-draws", "5000", "--output-dir", str(PREDICTION_DIR)]
    if PREDICTION_DIR.exists() and any(PREDICTION_DIR.iterdir()):
        command.append("--resume")
    subprocess.run(command, check=True, cwd=ROOT)
elif FORECAST_MODE == "run":
    print("Đang chờ BO hoàn tất; chưa chạy forecasting.")
else:
    print("Đọc kết quả từ:", PREDICTION_DIR)

prediction_metadata = read_json(PREDICTION_DIR / "metadata.json")
print("Prediction status:", prediction_metadata.get("status", "pending"))

## 6. Issued predictions và fitted uncertainty của bang được chọn

`predictions.csv` chỉ chứa dự báo đã phát hành; `quantiles.csv` chứa phân phối
NB sau median shift. MAE/WIS và ghép future truth nằm trong notebook evaluation.
Bảng có thể xem khi bang này đã xong dù các bang khác còn đang chạy.

In [ ]:
STATE_PREDICTION_DIR = None
index_path = PREDICTION_DIR / "series_metrics.csv"
if index_path.is_file():
    state_index = pd.read_csv(index_path, dtype={"output_subdir": str})
    found = state_index.loc[state_index["series"] == SELECTED_SERIES]
    if len(found) == 1:
        STATE_PREDICTION_DIR = PREDICTION_DIR / found.iloc[0]["output_subdir"]
elif (PREDICTION_DIR / "predictions.csv").is_file():
    payload = pd.read_csv(PREDICTION_DIR / "predictions.csv", nrows=1)
    if payload.iloc[0]["series"] == SELECTED_SERIES:
        STATE_PREDICTION_DIR = PREDICTION_DIR

issued = quantiles = calibration = None
if STATE_PREDICTION_DIR is not None:
    issued = pd.read_csv(STATE_PREDICTION_DIR / "predictions.csv", parse_dates=["forecast_origin", "target_end_date"])
    quantiles = pd.read_csv(STATE_PREDICTION_DIR / "quantiles.csv", parse_dates=["forecast_origin", "target_end_date"])
    calibration = pd.read_csv(STATE_PREDICTION_DIR / "calibration.csv")
    issued = issued.drop(columns=["baseline"], errors="ignore")
    quantiles = quantiles.loc[quantiles.model == "smoa"]
    calibration = calibration.loc[calibration.model == "smoa"]
    print(f"{SELECTED_SERIES}: {issued['forecast_origin'].nunique()} origins; {len(issued):,} issued forecasts")
    display(issued.tail(8))
    display(quantiles.tail(8))
    display(calibration.tail(8))
else:
    print(f"Chưa có issued predictions của {SELECTED_SERIES}; chạy lại cell sau khi job đi tới prediction.")

## 7. Một lần phát hành: history → dự báo 1–4 tuần

Mỗi đường dự báo ở hình này có **cùng một origin**, mặc định origin mới nhất.
Chỉ vẽ observed history tới origin và tương lai dự báo, không ghép actual tương lai
vào đồ thị phát hành. Xem đối chiếu forecast/actual trong `evaluation.ipynb`.

In [ ]:
if issued is not None:
    origin = issued["forecast_origin"].max()
    forecast = issued.loc[issued["forecast_origin"] == origin].sort_values("horizon")
    state_config = read_json(STATE_PREDICTION_DIR / "metadata.json").get("config", {})
    history = truth.loc[truth["target_end_date"] <= origin].copy()
    archive_path = state_config.get("as_of_truth")
    if archive_path:
        archive = pd.read_csv(archive_path, parse_dates=["as_of", "target_end_date"])
        history = archive.loc[(archive["as_of"] == origin) & (archive["target_end_date"] <= origin),
                              ["target_end_date", "value"]].sort_values("target_end_date")
    history = history.tail(12)
    future_q = quantiles.loc[(quantiles["forecast_origin"] == origin) & (quantiles["model"] == "smoa")]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(history["target_end_date"], np.maximum(history["value"], 1), color="#172537", marker="o", label="History at origin (floor 1)")
    for lower, upper, alpha, label in [(.025, .975, .12, "95% predictive interval"), (.25, .75, .25, "50% predictive interval")]:
        bounds=[]
        for horizon in forecast["horizon"]:
            group = future_q.loc[future_q["horizon"] == horizon]
            lo = group.loc[np.isclose(group["quantile"], lower), "value"]
            hi = group.loc[np.isclose(group["quantile"], upper), "value"]
            if len(lo) == 1 and len(hi) == 1:
                bounds.append((lo.iloc[0], hi.iloc[0]))
        if len(bounds) == len(forecast):
            ax.fill_between(forecast["target_end_date"], [b[0] for b in bounds], [b[1] for b in bounds],
                            color="#1768ac", alpha=alpha, label=label)
    ax.plot(forecast["target_end_date"], forecast["forecast"], color="#1768ac", marker="o", label="sMOA")
    ax.axvline(origin, color="#777777", linestyle=":", linewidth=1)
    ax.set(title=f"{SELECTED_SERIES} | Forecast origin {origin:%Y-%m-%d}", xlabel="Week ending", ylabel="Weekly reported cases")
    ax.legend(loc="best", fontsize=9); ax.grid(axis="y", alpha=.2)
    show_figure(fig)
else:
    print("Chưa có forecast để vẽ; không thay bằng dữ liệu minh họa.")

## Tiếp theo: evaluation

Mở **`evaluation.ipynb`**. Nếu forecasting chạy ở chế độ `run`, đổi
`PREDICTION_DIR` trong notebook evaluation sang `outputs/notebook_forecasting/full`,
rồi chọn `EVALUATION_MODE="recompute"` để chấm dự báo vừa tạo.
Final-truth history, SciPy smoothing và NumPy RNG vẫn có các khác biệt ghi trong
`REPRODUCTION_AUDIT.md`; chạy full ngân sách không đồng nghĩa khớp từng số với R.